# PurushTradingAIAgents — train the technical analyst (LightGBM) on free Colab/Kaggle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/PurushTradingAIAgents/blob/main/training/notebooks/train_gbm_colab.ipynb)

Walk-forward, purged cross-validation on NIFTY 50 daily bars from yfinance, triple-barrier labels, out-of-fold metrics, fee-aware backtest versus a NIFTY baseline, versioned model registry. Replace `YOUR_GITHUB_USER` with your GitHub user once the repo is pushed.

> Paper research only. Nothing here is investment advice.

In [ ]:
# 1. Install the package from GitHub (or upload the repo folder and `pip install -e .`)
!pip -q install "git+https://github.com/YOUR_GITHUB_USER/PurushTradingAIAgents.git#egg=ptaa[ml]"
import ptaa; print('ptaa', ptaa.__version__)

In [ ]:
# 2. Config and data directory (Colab's /content is ephemeral; mount Drive to keep models)
import os, pathlib
from google.colab import drive  # comment out on Kaggle
drive.mount('/content/drive')
DATA_DIR = pathlib.Path('/content/drive/MyDrive/ptaa-data'); DATA_DIR.mkdir(parents=True, exist_ok=True)
os.environ['PTAA_DATA_DIR'] = str(DATA_DIR)
!curl -sO https://raw.githubusercontent.com/YOUR_GITHUB_USER/PurushTradingAIAgents/main/config.yaml
from ptaa.settings import load_config
cfg = load_config('config.yaml'); cfg.universe

In [ ]:
# 3. Download NIFTY 50 daily history (cached under DATA_DIR) and the NIFTY index for the baseline
from datetime import datetime, timedelta
import pandas as pd, yfinance as yf
from ptaa.training.pipeline import yfinance_candle_source
YEARS = 12
source = yfinance_candle_source(DATA_DIR, years=YEARS)
end = datetime.now(); raw = yf.download('^NSEI', start=end - timedelta(days=365*YEARS), end=end, interval='1d', progress=False, auto_adjust=False, multi_level_index=False)
index_close = raw['Close'].rename('close'); index_close.index = pd.to_datetime(index_close.index).tz_localize('Asia/Kolkata')
source('RELIANCE').tail()

In [ ]:
# 4. Train: walk-forward purged CV, out-of-fold metrics, registry entry
import logging; logging.basicConfig(level='INFO')
from ptaa.mlmodels.registry import ModelRegistry
from ptaa.training.pipeline import run_training
registry = ModelRegistry(DATA_DIR / 'models')
rep = run_training(cfg, registry, source, horizon='5d', universe='NIFTY50', index_close=index_close, n_folds=5, min_train=20000)
rep.metrics

In [ ]:
# 5. Backtest versus baseline (fees + slippage included) and threshold sweep
print('strategy', rep.strategy); print('baseline', rep.baseline)
rep.sweep

In [ ]:
# 6. Explain the latest bar of one symbol with SHAP
from ptaa.fast.features import FEATURE_NAMES, compute_features
from ptaa.training.train_gbm import GBMModel
m = GBMModel(registry, rep.model_id)
row = compute_features(source('RELIANCE'), index_close).dropna(subset=[c for c in FEATURE_NAMES if not c.startswith(('rel_','beta'))]).iloc[-1]
print(m.predict_proba(row)); m.explain(row)

In [ ]:
# 7. Registry contents; copy models/ to GitHub release assets for the laptop
for r in registry.list(): print(r.model_id, r.train_end, r.metrics.get('balanced_accuracy'), registry.verify(r.model_id))
!ls -la {DATA_DIR}/models {DATA_DIR}/models/gbm

## Reading the results honestly

* `balanced_accuracy` near 0.33 on three classes means no edge; anything sustained above ~0.40 out-of-fold is interesting.
* Compare `strategy` to `baseline` after fees: a model that cannot beat buy-and-hold NIFTY after costs is not deployed.
* `top_decile_precision_up` is what the ensemble floor uses; the threshold sweep shows how fee drag changes with trade count.
* Retrain weekly; the `model_freshness` rule rejects stale models.